<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


### Hands On Lab - Saving and loading a SparkML model


#### Objectives:
In this lab you will
 - Create a simple Linear Regression Model
 - Save the SparkML model
 - Load the SparkML model
 - Make predictions using the loaded SparkML model


#### Install pyspark


In [1]:
# Dependencies are already installed and configured in the environment
# !pip install pyspark
# !pip install findspark


#### Import libraries


In [2]:
# Pre-configured pyspark 3.5.6 with Windows Hadoop binaries


In [1]:
import os, sys
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21"
os.environ["HADOOP_HOME"] = r"C:\Users\marwa\hadoop"
if r"C:\Users\marwa\hadoop\bin" not in os.environ.get("PATH", ""):
    os.environ["PATH"] = r"C:\Users\marwa\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import findspark
findspark.init()


In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession


#### Creating the spark session and context


In [3]:
# Creating a spark context class
sc = SparkContext.getOrCreate()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Saving and Loading a SparkML Model") \
    .getOrCreate()


#### Importing Spark ML libraries


In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression


#### Create a DataFrame with sample data


In [5]:
# Create a simple data set of infant height(cms) weight(kgs) chart.

mydata = [[46,2.5],[51,3.4],[54,4.4],[57,5.1],[60,5.6],[61,6.1],[63,6.4]]
  
# Mention column names of dataframe
columns = ["height", "weight"]
  
# creating a dataframe
mydf = spark.createDataFrame(mydata, columns)
  
# show data frame
mydf.show()


+------+------+
|height|weight|
+------+------+
|    46|   2.5|
|    51|   3.4|
|    54|   4.4|
|    57|   5.1|
|    60|   5.6|
|    61|   6.1|
|    63|   6.4|
+------+------+



#### Converting data frame columns into feature vectors

In this task we use the `VectorAssembler()` function to convert the dataframe columns into feature vectors. 
For our example, we use the horsepower ("hp) and weight of the car as input features and the miles-per-gallon ("mpg") as target labels.


In [6]:
assembler = VectorAssembler(
    inputCols=["height"],
    outputCol="features")

data = assembler.transform(mydf).select('features','weight')


In [7]:
data.show()


+--------+------+
|features|weight|
+--------+------+
|  [46.0]|   2.5|
|  [51.0]|   3.4|
|  [54.0]|   4.4|
|  [57.0]|   5.1|
|  [60.0]|   5.6|
|  [61.0]|   6.1|
|  [63.0]|   6.4|
+--------+------+



#### Create and Train model

We can create the model using the `LinearRegression()` class and train using the `fit()` function. 


In [9]:
# Create a LR model
lr = LinearRegression(featuresCol='features', labelCol='weight')
lrModel = lr.fit(data)

print(f"Coefficients: {lrModel.coefficients}")
print(f"Intercept: {lrModel.intercept}")


Coefficients: [0.2363636363636347]
Intercept: -8.450649350649257


#### Save the model


In [10]:
lrModel.write().overwrite().save('infantheight2.model')


#### Load the model


In [11]:
# You need LinearRegressionModel to load the model
from pyspark.ml.regression import LinearRegressionModel


In [12]:
model = LinearRegressionModel.load('infantheight2.model')


#### Make Prediction


#### Predict the weight of an infant whose height is 70 CMs.


In [13]:
# This function converts a scalar number into a dataframe that can be used by the model to predict.
def predict(height):
    assembler = VectorAssembler(inputCols=["height"], outputCol="features")  # Adjusted input column name
    data = [[height, 0]]  # Changed input to reflect height
    columns = ["height", "weight"]  # Updated column names for clarity
    df = spark.createDataFrame(data, columns)
    transformed_df = assembler.transform(df).select('features', 'weight')  # Updated column selection
    predictions = model.transform(transformed_df)
    predictions.select('prediction').show()


In [14]:
predict(70)


+-----------------+
|       prediction|
+-----------------+
|8.094805194805172|
+-----------------+



### Practice exercises


#### Save the model as `babyweightprediction.model`


In [16]:
lrModel.write().overwrite().save('babyweightprediction.model')


Double-click __here__ for the solution.

<!-- Hint:

lrModel.write().overwrite().save('babyweightprediction.model')

-->


#### Load the model `babyweightprediction.model`


In [17]:
model = LinearRegressionModel.load('babyweightprediction.model')


Double-click __here__ for the solution.

<!-- Hint:

model = LinearRegressionModel.load('babyweightprediction.model')
-->


#### Predict the weight of an infant whose height is 50 CMs.


In [18]:
predict(50)


+------------------+
|        prediction|
+------------------+
|3.3675324675324774|
+------------------+



Double-click __here__ for the solution.

<!-- Hint:

predict(50)
-->
